# 09 · 가상 3D 공간 배치 (실시간, 카메라 | 가상씬 합성)

검출한 물체를 **가상 3D 공간에 원통 프리미티브로 배치**하고, 카메라 화면과 **나란히** 보여준다.
매 프레임 재렌더 → **물체를 놓으면 생기고 치우면 사라진다**(실시간 add/remove).

**파이프라인:** DA 깊이(감지+3D점군) + FastSAM(실루엣) + ArUco(평면·좌표계) → 물체별 3D 점군 → **PCA로 방향성 원통**(center·axis·length·radius) 적합 → 가상 카메라(3/4 시점)로 렌더.

**측정(원통 적합, robust):** 립밤 len69/dia16(GT69/19), 마커 76/13(GT78/13), 볼펜 116/11(GT148/12) — 지금까지 중 가장 정확. (누운 물체 길이는 끝단 가림만큼 −)

> 향후 넓은 작업공간은 이 평면을 **분산 앵커(id0~30)** 로 대체하면 그대로 확장된다.

In [ ]:
import os, sys, glob
import cv2, numpy as np
import matplotlib.pyplot as plt
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(ROOT, 'src'))
import aruco_utils as au, depth_volume as dv, live_combined as lc, scene3d as s3
from ultralytics import FastSAM

OUTPUT_DIR = os.path.join(ROOT, 'output'); SCENE_DIR = os.path.join(ROOT, 'data', 'scene_images')
SQ = 0.038
board = cv2.aruco.CharucoBoard((5, 7), SQ, SQ*22/30, cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_5X5_1000))
K, dist = au.load_intrinsics(os.path.join(OUTPUT_DIR, 'camera_intrinsics.npz'))
CAM_INDEX = 0
pipe = dv.load_depth_model('depth-anything/Depth-Anything-V2-Small-hf', device=0)
model = FastSAM('FastSAM-s.pt')
print('ready')

## 정지영상 미리보기 (카메라 합성 | 가상 3D 씬)

In [ ]:
cands = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'snap_raw_*.png'))) or sorted(glob.glob(os.path.join(SCENE_DIR, '*.*')))
img = cv2.imread(cands[-1]); print('대상:', os.path.basename(cands[-1]))
vis, objs, markers = lc.process_frame_combined(img, pipe, model, board, K, dist, imgsz=1024)
print(f"물체 {len(objs)}개, 마커 {len(markers)}개")
for i, o in enumerate(objs):
    cyl = o.get('cyl')
    if cyl: print(f"  #{i} {o['type']:5s} len={cyl[2]:.0f} dia={2*cyl[3]:.0f}mm  center={tuple(round(v) for v in cyl[0])}")
scene = s3.render_virtual_scene(objs, markers=markers, ws=(240, 320))
camh = cv2.resize(vis, (int(vis.shape[1]*scene.shape[0]/vis.shape[0]), scene.shape[0]))
plt.figure(figsize=(18, 7)); plt.imshow(cv2.cvtColor(np.hstack([camh, scene]), cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

In [ ]:
# 인터랙티브 3D 뷰어 (마우스로 회전/확대/이동) — plotly
# 노트북 안에서 fig.show()로 조작하거나, output/virtual_scene.html 로 저장해 브라우저에서 열기.
fig = s3.render_plotly(objs, markers=markers, html_path=os.path.join(OUTPUT_DIR, 'virtual_scene.html'))
fig.show()
print('저장: output/virtual_scene.html (브라우저에서 열면 마우스로 회전/확대)')

## 실시간 (카메라 | 가상 3D 씬 나란히)

보드 위에 물체를 올리면 오른쪽 가상 씬에 원통으로 뜨고, 치우면 사라진다. **`s`**=스냅, **`q`**=종료.

In [ ]:
lc.run_live_combined(K, dist, board, square_len=SQ, pipe=pipe, model=model,
                     cam_index=CAM_INDEX, imgsz=640, snapshot_dir=OUTPUT_DIR)
print('종료됨')

## 튜닝 & 다음

- **느리면**: `imgsz`↓. DA+FastSAM 2모델이라 GTX1660 수 fps.
- **원통 방향이 이상**: 점군 노이즈 → `fit_cylinder`의 이상치 컷(percentile) 조정.
- **가상씬 시점**: `render_virtual_scene(az, el)`로 회전각 조절.
- 다음: 분산 앵커(id0~30)로 평면 대체(넓은 작업공간), 꺾임 판별 개선, 물체 ID/추적으로 add/remove 안정화.